# exp083 — TTA espejo a presupuesto constante sobre exp080 (fold 0, sin reentrenar)

Padre: exp080_seqhead_ctrl_mil (backbone exp074 `swa_f0.pt` congelado + cabeza MIL reentrenada `swa_mil_f0.pt`).
Única variable: de las 64 ventanas que usa el kernel, las de posición impar entran espejadas con la convención de
entrenamiento de exp045 (sagital → invertir los 3 cortes-canal; coronal → volteo W; etiquetas sin permutar). No se
entrena nada: embeddings con y sin espejo, la cabeza de exp080 tal cual, pseudo L2 fold 0 decide, gold solo veta.
Ver SPEC.md.

Celdas fijas: **0** parámetros · **1** setup · **2** datos y split · **3** modelo · **4** embeddings (sin entrenamiento) ·
**5** evaluación · **6** guardar · **7** verdict.


## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp083_flip_tta_alterno"
PADRE = "exp080"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True
ALL_DATA = False          # True: entrena con los 4349 no-gold, sin validación; SWA sobre SWA_EPOCHS fijas
SWA_EPOCHS = []           # solo ALL_DATA: épocas (0-index) a promediar, fijadas con lo aprendido en fold 0

CORPUS = "/workspace/data/cache/raptor96_384"   # seqhead: el mismo corpus (build_corpus96.py) leído del volumen; solo se lee una vez para los embeddings
LABELS_SOFT = "/workspace/data/labels/public/dreaddevelopment/rsna-knee-labels/labels_llm_soft.parquet"  # fuente 1 (la del padre)
L3_CSV = "/workspace/data/labels/public/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"  # exp031: fuente 3
L4_CSV = "/workspace/data/labels/public/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv"  # exp031: fuente 4
L2_CSV = "/workspace/data/labels/public/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"  # vara y fuente 2
V1_CSV = "/workspace/data/labels/report_labels_v1.csv"
TEACHER_CSV = "/workspace/users/rodrigo/wt_exp074_raptor224_distill/results/exp074_raptor224_distill/teacher.csv"  # exp074: media en prob. 0.5/0.5 R7 compact + nartaa 384 (fuera de git)
TEACHER_W = 0.5          # exp074: objetivo = TEACHER_W × teacher + (1 − TEACHER_W) × consenso exp031
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
IMG = 384                 # lado guardado en el corpus
RES = 320                 # recorte central RES del corte IMG
RES_IN = 224              # exp057: el recorte RES se reescala a RES_IN (bilineal, align_corners=False)
USE_SLOTS = [0, 1, 2]     # exp057: Sag fluido, Sag, Cor fluido (cortes 0–65)
NORM = "imagenet"
L2_THRESHOLD = 0.5
N_WORKERS = 12
BACKBONE_CKPT = "/workspace/users/rodrigo/wt_exp074_raptor224_distill/results/exp074_raptor224_distill/swa_f0.pt"   # exp074 fold 0
HEAD_CKPT = "/workspace/users/rodrigo/wt_seqhead_c/results/exp080_seqhead_ctrl_mil/swa_mil_f0.pt"                  # exp080 cabeza MIL fold 0 (SWA)
K_SEQ = 64                 # ventanas por estudio (= K_EVAL del kernel), ordenadas por corte
EMB_BATCH = 256            # ventanas por forward del backbone
MIRROR_PHASE = 1           # exp083: posiciones j % 2 == MIRROR_PHASE de las 64 entran espejadas (la variable)
BOOT_N = 1000              # bootstrap pareado del Δ pseudo (informativo)
CRIT_WORKED = 0.002        # SPEC: Δ pseudo f0 (variante − control mismo pipeline) >= +0.002
CRIT_KILLED = 0.02         # SPEC: killed si gold variante < gold control − 0.02
CRIT_DOUBLE = 0.004        # SPEC: 64+64 (informativo) solo interesa si Δ pseudo >= +0.004
REPRO_TOL = 0.0005         # control debe reproducir exp080 SWA f0 (0.9018 / 0.9252) dentro de esta tolerancia
NOTES = "exp083: evaluación sin entrenamiento; 64 ventanas, impares espejadas (convención exp045, etiquetas sin permutar); cabeza exp080 f0 sin reentrenar"


## 1. Setup

In [ ]:
import copy
import json
import math
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · "
      f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'} · timm {timm.__version__} · torch {torch.__version__}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
def _read(path, cols=None):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    return df.set_index("StudyInstanceUID")[cols or LABELS].astype(float)

split = load_split(SPLIT)
def _read_soft(path, name):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, low_memory=False)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    assert not df.StudyInstanceUID.duplicated().any(), f"{name} con StudyInstanceUID duplicados"
    df = df.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce").astype(float)
    assert df.stack().between(0.0, 1.0).all(), f"{name} fuera de [0, 1]"
    return df

# exp031: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas (si falta una fuente, media de las presentes)
SOURCES = {"raptor": _read_soft(LABELS_SOFT, "raptor"), "L2": _read_soft(L2_CSV, "L2"),
           "L3": _read_soft(L3_CSV, "L3"), "L4": _read_soft(L4_CSV, "L4")}
_all = sorted(set().union(*[s.index for s in SOURCES.values()]))
_stack = np.stack([s.reindex(_all)[LABELS].to_numpy(dtype=float) for s in SOURCES.values()])
_n_src = np.isfinite(_stack).sum(0)
soft = pd.DataFrame(np.where(_n_src > 0, np.nansum(_stack, 0) / np.maximum(_n_src, 1), np.nan),
                    index=pd.Index(_all, name="StudyInstanceUID"), columns=LABELS)
N_SRC = pd.DataFrame(_n_src, index=soft.index, columns=LABELS)
l2 = _read(L2_CSV)                    # vara interna: L2 binarizada > 0.5
v1 = _read(V1_CSV)                    # informativo
train_csv = pd.read_csv("/workspace/data/raw/train.csv")
train_csv["StudyInstanceUID"] = train_csv.StudyInstanceUID.astype(str)
gold = train_csv.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")[LABELS].astype(float)
gold_ids = list(gold.index)
assert len(gold) == 58 and set(gold_ids) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
soft = soft.dropna(how="all")         # estudio sin ninguna fuente fuera; celda sin fuentes (NaN) se enmascara en la pérdida
assert np.nanmin(soft.values) >= 0 and np.nanmax(soft.values) <= 1
print(f"etiquetas de entrenamiento (consenso): {len(soft)} estudios · celdas NaN {int(soft.isna().sum().sum())}")

# exp074: destilación. Fila del teacher con NaN = estudio que el teacher no puntuó -> se queda el consenso solo.
teacher = pd.read_csv(TEACHER_CSV)
teacher["StudyInstanceUID"] = teacher.StudyInstanceUID.astype(str)
assert not teacher.StudyInstanceUID.duplicated().any()
teacher = teacher.set_index("StudyInstanceUID")[LABELS].astype(float)
_tv = teacher.to_numpy()[np.isfinite(teacher.to_numpy())]
assert ((_tv >= 0) & (_tv <= 1)).all()
_ng = [u for u in soft.index if u not in set(gold_ids)]
_miss = [u for u in _ng if u not in teacher.index]
assert not _miss, f"teacher sin {len(_miss)} estudios de train"
_t = teacher.reindex(soft.index)
_tok = _t.notna().all(axis=1)
# R7 deja sin puntuar algunos estudios grandes (fallback de su worker, igual que en Kaggle): ahí queda el consenso solo
assert (~_tok.loc[_ng]).mean() < 0.10, f"teacher falló en {int((~_tok.loc[_ng]).sum())} estudios"
SOFT_CONSENSO = soft.copy()
_m = _tok.to_numpy()[:, None]
soft = pd.DataFrame(np.where(_m, TEACHER_W * _t.to_numpy() + (1 - TEACHER_W) * soft.to_numpy(), soft.to_numpy()),
                    index=soft.index, columns=LABELS)
print(f"teacher: {len(teacher)} estudios · sin puntuar en train {int((~_tok.loc[_ng]).sum())} · "
      f"media teacher {np.nanmean(_t.loc[_ng].to_numpy()):.3f} · consenso {np.nanmean(SOFT_CONSENSO.loc[_ng].to_numpy()):.3f} · "
      f"objetivo {np.nanmean(soft.loc[_ng].to_numpy()):.3f}")
print("acuerdo binarizado teacher–consenso (> 0.5) por hallazgo: " + " · ".join(
    f"{c} {np.nanmean((_t.loc[_ng, c] > 0.5) == (SOFT_CONSENSO.loc[_ng, c] > 0.5)):.3f}" for c in LABELS))

corpus_ids = np.load(f"{CORPUS}/ids.npy").astype(str)
corpus_masks = np.load(f"{CORPUS}/masks.npy")
ID2ROW = {u: i for i, u in enumerate(corpus_ids)}
D = corpus_masks.shape[1]
assert D == 96, D
meta = json.load(open(f"{CORPUS}/build_meta.json"))
print(f"corpus {len(corpus_ids)} × {D} · cortes válidos medios {corpus_masks.sum(1).mean():.1f} · "
      f"estudios vacíos {(corpus_masks.sum(1) == 0).sum()} · errores build {len(meta['errors'])}")
assert set(gold_ids) <= set(ID2ROW)

pseudo_bin = (l2 > L2_THRESHOLD).astype(float).where(l2.notna())
v1_bin = v1.where(v1.isin([0.0, 1.0]))

def fold_ids(fold):
    """train / val para un fold de split_v1; train = pool sin el fold ∩ etiquetas de entrenamiento (consenso)."""
    pool = split[split.fold >= 0]
    tr = [u for u in pool.loc[pool.fold != fold, "StudyInstanceUID"] if u in soft.index and u in ID2ROW]
    va = [u for u in pool.loc[pool.fold == fold, "StudyInstanceUID"] if u in ID2ROW]
    return tr, va

for f in FOLDS:
    tr, va = fold_ids(f)
    assert not set(tr) & set(gold_ids), "gold dentro del entrenamiento"
    assert not set(tr) & set(va)
    print(f"fold {f}: train {len(tr)} · val {len(va)} (L2 cubre {pseudo_bin.index.isin(va).sum()}) · gold {len(gold_ids)}")
    _ns = N_SRC.reindex(tr).fillna(0).to_numpy()
    print(f"  celdas de train por nº de fuentes (0..4): {[int((_ns == k).sum()) for k in range(5)]} · "
          f"cobertura por fuente (estudios de train): {({n: int(pd.Index(tr).isin(s.index).sum()) for n, s in SOURCES.items()})}")
    _tr = {n: s.reindex(tr)[LABELS].to_numpy() for n, s in SOURCES.items()}
    _agree = lambda a, b: np.mean([(x > 0.5) == (y > 0.5) for x, y in zip(_tr[a].ravel(), _tr[b].ravel())
                                   if np.isfinite(x) and np.isfinite(y)])
    print("  acuerdo binarizado (> 0.5) en train: " + " · ".join(f"{a}–{b} {_agree(a, b):.3f}" for a, b in
          [("L2", "L4"), ("L2", "L3"), ("L3", "L4"), ("raptor", "L2"), ("raptor", "L3"), ("raptor", "L4")]))
    print(f"  media del objetivo en train: consenso {np.nanmean(soft.loc[tr].to_numpy()):.3f} · " +
          " · ".join(f"{n} {np.nanmean(v):.3f}" for n, v in _tr.items()))

## 3. Modelo (verbatim del script público: `build_backbone` + `RaptorClassifier`; dataset `StudyWindows`)

In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
SLOT_SIZES = [26, 22, 18, 12, 18]     # exp045: slots del corpus (Sag fluid, Sag, Cor fluid, Cor, Ax), como el TTA nartaa
SAGITTAL_SLOTS = (0, 1)
SLOT_OF = np.repeat(np.arange(len(SLOT_SIZES)), SLOT_SIZES)
assert len(SLOT_OF) == D
IN_SLOTS = np.isin(SLOT_OF, USE_SLOTS)      # exp057: cortes de los slots usados


class StudyWindows(Dataset):
    """Bolsa de ventanas 2.5D por estudio (3 cortes consecutivos -> RGB) desde el corpus memmap.
    Igual que el script público salvo: recorte central RES del corte IMG (en vez de resize) y la conversión a float,
    la ganancia ×U(0.9,1.1) y la normalización ImageNet se hacen en GPU (mismas operaciones)."""
    def __init__(self, ids, k, train):
        self.ids, self.k, self.train = list(ids), k, train
        self.vols = None

    def __len__(self):
        return len(self.ids)

    def _centers(self, valid, count):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, D - 2))]
        if self.train:
            pool = cs * (count // len(cs) + 1)
            random.shuffle(pool)
            return pool[:count]
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return [cs[i] for i in idx]

    def __getitem__(self, i):
        if self.vols is None:
            self.vols = np.load(f"{CORPUS}/vols.npy", mmap_mode="r")
        uid = self.ids[i]; row = ID2ROW[uid]
        valid = np.where((corpus_masks[row] > 0) & IN_SLOTS)[0]     # exp057: solo slots USE_SLOTS
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        cs = np.array([max(1, min(c, D - 2)) for c in self._centers(valid, self.k)])
        lo = (IMG - RES) // 2
        vol = self.vols[row]
        tri = np.stack([vol[cs - 1, lo:lo + RES, lo:lo + RES], vol[cs, lo:lo + RES, lo:lo + RES],
                        vol[cs + 1, lo:lo + RES, lo:lo + RES]], 1)          # (K,3,RES,RES) uint8
        gain = 1.0 + (random.random() - 0.5) * 0.20 if self.train else 1.0
        sag = torch.from_numpy(np.isin(SLOT_OF[cs], SAGITTAL_SLOTS))     # exp045: eje del espejo por ventana (slot del corte central)
        return torch.from_numpy(np.ascontiguousarray(tri)), torch.tensor(gain, dtype=torch.float32), i, sag


def spatial_aug(x, sag):
    """exp045: espejo L/R anatómico (p AUG_FLIP_P) + afín (misma matriz para las K ventanas de un estudio), en GPU.
    Espejo = `_mirror_tri` del TTA nartaa: ventana con corte central sagital -> invertir los 3 cortes-canal; si no -> volteo W."""
    B, K_, C, H, W = x.shape
    flip = (torch.rand(B, device=x.device) < AUG_FLIP_P).view(B, 1, 1, 1, 1)
    sag = sag.to(x.device).view(B, K_, 1, 1, 1)
    x = torch.where(flip, torch.where(sag, x.flip(2), x.flip(-1)), x)
    ang = (torch.rand(B, device=x.device) * 2 - 1) * math.radians(AUG_ROT_DEG)
    scl = AUG_SCALE[0] + torch.rand(B, device=x.device) * (AUG_SCALE[1] - AUG_SCALE[0])
    tr = (torch.rand(B, 2, device=x.device) * 2 - 1) * AUG_TRANS * 2      # coords normalizadas [-1, 1]: lado = 2
    cos, sin = torch.cos(ang) / scl, torch.sin(ang) / scl                  # theta: salida -> entrada (escala > 1 = zoom in)
    theta = torch.stack([torch.stack([cos, -sin, tr[:, 0]], 1), torch.stack([sin, cos, tr[:, 1]], 1)], 1)  # (B,2,3)
    theta = theta.repeat_interleave(K_, 0)
    flat = x.reshape(B * K_, C, H, W)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    return F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False).view(B, K_, C, H, W)


def to_input(x_u8, gain, sag=None, train=False):
    x = x_u8.to(DEVICE, non_blocking=True).float() / 255.0
    if train:
        x = spatial_aug(x, sag)                                            # exp045: solo entrenamiento
    if RES_IN != x.shape[-1]:                                              # exp057: 320 -> 224 antes de normalizar
        B_, K__ = x.shape[:2]
        x = F.interpolate(x.flatten(0, 1), size=(RES_IN, RES_IN), mode="bilinear", align_corners=False).view(B_, K__, 3, RES_IN, RES_IN)
    if not torch.all(gain == 1.0):
        x = (x * gain.to(DEVICE).view(-1, 1, 1, 1, 1)).clamp(0, 1)
    if NORM == "imagenet":
        x = (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)
    return x


def build_backbone(arch, pretrained=True):
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, in_chans=3, global_pool="avg", img_size=RES_IN)   # exp057


class RaptorClassifier(nn.Module):
    """Raptor encoder + per-diagnosis attention-MIL head (12 findings)."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K_ = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).view(B, K_, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


def build_model():
    bb = build_backbone(ARCH, pretrained=True)
    if GRAD_CKPT:
        bb.set_grad_checkpointing(True)
    return RaptorClassifier(bb, F_dim=bb.num_features).to(DEVICE)



# ---- exp083: backbone congelado -> embeddings de las 64 ventanas SIN espejo y ESPEJADAS (convención spatial_aug de exp045) ----
def mirror_windows(x_u8, sag):
    """Espejo L/R anatómico de exp045 (`spatial_aug` con flip = 1): ventana con corte central sagital -> invertir los 3
    cortes-canal (dim 2); si no -> volteo del ancho. Etiquetas sin permutar. x (B,K,3,H,W) sobre el recorte RES."""
    s = sag.to(x_u8.device).view(x_u8.shape[0], x_u8.shape[1], 1, 1, 1)
    return torch.where(s, x_u8.flip(2), x_u8.flip(-1))


def load_frozen(ckpt_path):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    bb = build_backbone(ARCH, pretrained=False)
    m = RaptorClassifier(bb, F_dim=bb.num_features)
    m.load_state_dict(ck["model"], strict=True)
    for p_ in m.parameters():
        p_.requires_grad_(False)
    return m.to(DEVICE).eval()


@torch.no_grad()
def embed_both(ckpt_path, ids):
    """-> (P, M) (len(ids), K_SEQ, F) float16: backbone congelado sobre las K_SEQ ventanas de evaluación de cada estudio
    (StudyWindows train=False = ventanas del kernel), sin espejo (P) y espejadas (M). bf16 autocast como exp080."""
    model = load_frozen(ckpt_path)
    F_ = model.backbone.num_features
    P_ = np.zeros((len(ids), K_SEQ, F_), np.float16); M_ = np.zeros_like(P_)
    dl = DataLoader(StudyWindows(ids, K_SEQ, False), batch_size=max(1, EMB_BATCH // (2 * K_SEQ)), shuffle=False,
                    num_workers=N_WORKERS, pin_memory=True)
    t0 = time.time(); n, nxt = 0, 200
    for x, g, idx, sag in dl:
        x = x.to(DEVICE, non_blocking=True)
        xb = torch.cat([x, mirror_windows(x, sag)], 1)                  # (B, 2K, 3, RES, RES) uint8
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            f = model.encode(to_input(xb, g)).float()
        f = f.cpu().numpy().astype(np.float16)
        ii = idx.numpy()
        P_[ii], M_[ii] = f[:, :K_SEQ], f[:, K_SEQ:]
        n += len(ii)
        if n >= nxt:
            nxt += 200
            print(f"  embeddings {n}/{len(ids)} · {(time.time() - t0) / n:.3f} s/estudio", flush=True)
    secs = time.time() - t0
    print(f"embeddings sin espejo + espejados: {len(ids)} estudios en {secs:.0f} s · F {F_}", flush=True)
    del model
    torch.cuda.empty_cache()
    return torch.from_numpy(P_), torch.from_numpy(M_), secs


class MILHead(nn.Module):
    """La cabeza de exp080 (`RaptorClassifier.head`) sin backbone; mismos nombres de parámetros."""
    def __init__(self, F_dim, n=12, drop=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))

    def forward(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb


## 4. Entrenamiento (bucle del script público; selección y top-K por pseudo L2 en val, nunca gold)

In [ ]:
def score(pred):
    r = {}
    for name, ref in (("pseudo", pseudo_bin), ("v1", v1_bin), ("gold", gold)):
        ids = [u for u in pred.index if u in ref.index]
        r[name] = mean_auc(ref.loc[ids, LABELS].to_numpy(), pred.loc[ids, LABELS].to_numpy()) if ids else None
    return r


# exp083: no se entrena nada. Embeddings sin espejo y espejados de val fold 0 + gold, una vez.
T_CELL4 = time.time()
_, VA_IDS = fold_ids(FOLDS[0])
if SMOKE:
    VA_IDS = VA_IDS[:48]
IDS = list(dict.fromkeys(VA_IDS + gold_ids))
EMB_P, EMB_M, EMB_S = embed_both(BACKBONE_CKPT, IDS)
EMB_P, EMB_M = EMB_P.to(DEVICE), EMB_M.to(DEVICE)
assert torch.isfinite(EMB_P.float()).all() and torch.isfinite(EMB_M.float()).all(), "embeddings con NaN/inf"
F_EMB = int(EMB_P.shape[-1])
_hk = torch.load(HEAD_CKPT, map_location="cpu", weights_only=False)
assert _hk["kind"] == "mil" and int(_hk["k_seq"]) == K_SEQ, (_hk["kind"], _hk["k_seq"])
HEAD = MILHead(F_EMB).to(DEVICE).eval()
HEAD.load_state_dict(_hk["head"], strict=True)
del _hk


@torch.no_grad()
def head_prob(feats):
    out = []
    for i in range(0, len(feats), 256):
        out.append(torch.sigmoid(HEAD(feats[i:i + 256].float())).cpu().numpy())
    return np.concatenate(out)


ALT = EMB_P.clone(); ALT[:, MIRROR_PHASE::2] = EMB_M[:, MIRROR_PHASE::2]            # LA variante: impares espejadas
ALT_C = EMB_P.clone(); ALT_C[:, 1 - MIRROR_PHASE::2] = EMB_M[:, 1 - MIRROR_PHASE::2]  # informativo: fase complementaria
PROBS = {
    "control_64": head_prob(EMB_P),
    "alt64_phase%d" % MIRROR_PHASE: head_prob(ALT),
    "info_alt64_phase%d" % (1 - MIRROR_PHASE): head_prob(ALT_C),
    "info_mirror64": head_prob(EMB_M),
    "info_mean_64plus64": (head_prob(EMB_P) + head_prob(EMB_M)) / 2,
    "info_bag128": head_prob(torch.cat([EMB_P, EMB_M], 1)),
    "info_two32_mean": (head_prob(EMB_P[:, 0::2]) + head_prob(EMB_M[:, 0::2])) / 2,
    "info_plain32": head_prob(EMB_P[:, 0::2]),
}
DECIDE = "alt64_phase%d" % MIRROR_PHASE
del ALT, ALT_C
GPU_H_TOTAL = (time.time() - T_CELL4) / 3600
print(f"celda 4: {GPU_H_TOTAL:.3f} GPU-h (embeddings {EMB_S:.0f} s) · {len(IDS)} estudios")


## 5. Evaluación (pseudo L2 decide; v1 y gold informativos/veto)

In [ ]:
def as_df(p):
    return pd.DataFrame(p, index=pd.Index(IDS, name="StudyInstanceUID"), columns=LABELS)


RES_V = {}
for k, p in PROBS.items():
    dv, dg = as_df(p).loc[VA_IDS], as_df(p).loc[gold_ids]
    sv, sg = score(dv), score(dg)
    RES_V[k] = {"pseudo_auc": sv["pseudo"]["mean"], "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
                "gold_auc": sg["gold"]["mean"], "pseudo_per_label": sv["pseudo"]["per_label"],
                "gold_per_label": sg["gold"]["per_label"]}
summary = pd.DataFrame({k: {m: v[m] for m in ("pseudo_auc", "v1_auc", "gold_auc")} for k, v in RES_V.items()}).T
ctrl = RES_V["control_64"]
summary["d_pseudo_vs_control"] = summary["pseudo_auc"] - ctrl["pseudo_auc"]
summary["d_gold_vs_control"] = summary["gold_auc"] - ctrl["gold_auc"]
display(summary.round(4))

# bootstrap pareado (estudios de val con L2) del Δ pseudo variante − control: informativo
_ref = pseudo_bin.reindex(VA_IDS)[LABELS].to_numpy()
_pv = {k: as_df(PROBS[k]).loc[VA_IDS, LABELS].to_numpy() for k in (DECIDE, "control_64", "info_mean_64plus64")}
rng = np.random.default_rng(0)
BOOT = {}
for k in (DECIDE, "info_mean_64plus64"):
    ds = []
    for _ in range(BOOT_N if not SMOKE else 50):
        b = rng.integers(0, len(VA_IDS), len(VA_IDS))
        ds.append(mean_auc(_ref[b], _pv[k][b])["mean"] - mean_auc(_ref[b], _pv["control_64"][b])["mean"])
    ds = np.array(ds)
    BOOT[k] = {"mean": float(ds.mean()), "ci95": [float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))],
               "p_gt0": float((ds > 0).mean())}
print("bootstrap Δ pseudo:", json.dumps(BOOT))

parent = leer_metricas(PADRE) or {}
P_SWA = parent.get("swa_f0") or {}
REPRO = {"pseudo": ctrl["pseudo_auc"] - (P_SWA.get("pseudo_auc") or float("nan")),
         "gold": ctrl["gold_auc"] - (P_SWA.get("gold_auc") or float("nan"))}
print(f"control vs exp080 SWA f0 (reproducción): Δ pseudo {REPRO['pseudo']:+.5f} · Δ gold {REPRO['gold']:+.5f}")
REPRO_OK = bool(abs(REPRO["pseudo"]) <= REPRO_TOL and abs(REPRO["gold"]) <= REPRO_TOL)
print("REPRO_OK", REPRO_OK, "(si False: el verdict queda en repro_fail, parar y avisar)")

var = RES_V[DECIDE]
delta_pseudo = var["pseudo_auc"] - ctrl["pseudo_auc"]
delta_gold = var["gold_auc"] - ctrl["gold_auc"]
if var["gold_auc"] < ctrl["gold_auc"] - CRIT_KILLED:
    VERDICT_AUTO = "killed"
elif delta_pseudo >= CRIT_WORKED:
    VERDICT_AUTO = "worked"
else:
    VERDICT_AUTO = "didnt"
if not REPRO_OK and not SMOKE:
    VERDICT_AUTO = "repro_fail"
d_double = RES_V["info_mean_64plus64"]["pseudo_auc"] - ctrl["pseudo_auc"]
DOUBLE_OK = bool(d_double >= CRIT_DOUBLE and RES_V["info_mean_64plus64"]["gold_auc"] >= ctrl["gold_auc"] - CRIT_KILLED)
DECISION = decision(delta_pseudo, delta_gold)   # regla genérica del equipo (informativa; decide la SPEC)
print(f"{DECIDE}: Δ pseudo {delta_pseudo:+.4f} · Δ gold {delta_gold:+.4f} -> VERDICT_AUTO {VERDICT_AUTO} | "
      f"64+64 (informativo): Δ pseudo {d_double:+.4f} -> cumple +{CRIT_DOUBLE}: {DOUBLE_OK} | genérica {DECISION}")


## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else "screen",
    "select_on": "pseudo_L2_bin>0.5 (sin selección: evaluación pura)", "params": {k: v for k, v in globals().items()
        if k.isupper() and isinstance(v, (int, float, str, list, bool)) and not (isinstance(v, list) and len(v) > 20)},
    "repro_ok": REPRO_OK,
    "decided_variant": DECIDE,
    "swa_f0": {"pseudo_auc": var["pseudo_auc"], "v1_auc": var["v1_auc"], "gold_auc": var["gold_auc"],
               "pseudo_per_label": var["pseudo_per_label"], "gold_per_label": var["gold_per_label"]},
    "gold_auc": var["gold_auc"], "oof_pseudo_auc": None,
    "control_swa_f0": {k: ctrl[k] for k in ("pseudo_auc", "v1_auc", "gold_auc")},
    "variants": RES_V, "bootstrap_delta_pseudo": BOOT,
    "parent_swa_f0": {"pseudo_auc": P_SWA.get("pseudo_auc"), "gold_auc": P_SWA.get("gold_auc")},
    "repro_control_vs_parent": REPRO,
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold, "pseudo_f0_double_64plus64": d_double},
    "double_64plus64_meets_crit": DOUBLE_OK,
    "decision_auto": DECISION, "verdict_auto": VERDICT_AUTO,
    "criteria": {"worked": f"delta pseudo f0 (variante - control) >= {CRIT_WORKED}",
                 "killed": f"gold variante < gold control - {CRIT_KILLED}",
                 "double_info": f"64+64 delta pseudo >= {CRIT_DOUBLE}"},
    "backbone_ckpt": BACKBONE_CKPT, "head_ckpt": HEAD_CKPT,
    "embeddings": {"seconds": EMB_S, "n_studies": len(IDS), "n_val": len(VA_IDS), "k_seq": K_SEQ, "F": F_EMB},
    "gpu_hours_total": GPU_H_TOTAL,
    "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu",
}
print("Guardado:", guardar_metricas(RESULTS, metrics))


## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza la fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed`). Después: commit y PR a `main`.